# 05 · Sending & Reading Emails (`smtplib`, `email`, `imaplib`)

**Module 14 – Working with Data & the Web** · Estimated time: 45 min

> Updates 📗 *Overview of Sending Emails* and *Overview of Received Emails*. ⚠️ The original builds messages by string concatenation (`"Subject: " + subject + "\n" + message`), which breaks with non-ASCII text, has no proper headers and can't do attachments. ✅ This lesson uses the modern **`email.message.EmailMessage`** API, secure TLS contexts and `with` blocks.

### 🎯 What you will learn
- How email sending works (SMTP) and the server/port table (📗)
- App passwords and why you must never hard-code credentials
- ✅ Building messages with `EmailMessage`: plain text, HTML, attachments
- Sending securely with `SMTP` + `starttls()` (port 587) or `SMTP_SSL` (port 465)
- Reading & searching mail with `imaplib` (📗 search keys) and parsing it with `email`
- Testing safely without sending real email

> 📧 Cells that contact a real mail server are **not pre-run** (they need your credentials). Everything else runs offline.

## 1. How it works

- **SMTP** (*Simple Mail Transfer Protocol*): **sending** mail
- **IMAP**: **reading** mail stored on the server (POP3 is older and downloads/deletes)

| Provider | SMTP server | Port | IMAP server |
|---|---|---|---|
| Gmail | `smtp.gmail.com` | 587 (STARTTLS) / 465 (SSL) | `imap.gmail.com` |
| Outlook / Microsoft 365 | `smtp.office365.com` / `smtp-mail.outlook.com` | 587 | `outlook.office365.com` |
| Yahoo | `smtp.mail.yahoo.com` | 465 / 587 | `imap.mail.yahoo.com` |
| Zoho | `smtp.zoho.com` | 465 / 587 | `imap.zoho.com` |

- **Port 587** → connect in plain text, then upgrade with **`starttls()`**
- **Port 465** → encrypted from the start with **`SMTP_SSL`**

### Credentials (📗, updated)
- **Gmail**: turn on 2-Step Verification, then create an **App Password** (Google Account → Security → App passwords) and use it instead of your normal password. (Google removed "less secure app access" in 2022.)
- **Microsoft 365**: basic-auth SMTP is being phased out for many tenants in favour of OAuth 2.0. Check your organisation's settings.
- ✅ Load credentials from **environment variables** (or a secrets manager); don't type them into notebooks that you might share, and never commit them to Git.

```python
import os
EMAIL = os.environ["EMAIL_USER"]
PASSWORD = os.environ["EMAIL_APP_PASSWORD"]
```
(📗 shows `getpass.getpass()`, which is fine for interactive use.)

## 2. ✅ Building a message with `EmailMessage`

In [1]:
from email.message import EmailMessage
from email.utils import formataddr, make_msgid, formatdate

msg = EmailMessage()
msg["Subject"] = "Your weekly report 📊"
msg["From"] = formataddr(("Python Bot", "bot@example.com"))
msg["To"] = "asha@example.com, ravi@example.com"
msg["Cc"] = "manager@example.com"
msg["Date"] = formatdate(localtime=True)
msg["Message-ID"] = make_msgid(domain="example.com")
msg.set_content("Hi team,\n\nThe weekly report is attached.\n\n— Python Bot")   # plain-text body

print(msg)

Subject: Your weekly report 📊
From: Python Bot <bot@example.com>
To: asha@example.com, ravi@example.com
Cc: manager@example.com
Date: Tue, 20 Oct 2026 19:46:02 +0530
Message-ID: <179250576222.6373.4525637047925896250@example.com>
Content-Type: text/plain; charset="utf-8"
Content-Transfer-Encoding: base64
MIME-Version: 1.0

SGkgdGVhbSwKClRoZSB3ZWVrbHkgcmVwb3J0IGlzIGF0dGFjaGVkLgoK4oCUIFB5dGhvbiBCb3QK



### HTML body with a plain-text fallback

In [2]:
msg.add_alternative("""\
<html><body>
  <p>Hi team,</p>
  <p>The <b>weekly report</b> is attached. Sales are <span style="color:green">up 12%</span>.</p>
  <p>— Python Bot</p>
</body></html>
""", subtype="html")
print(msg.get_content_type())                     # multipart/alternative

multipart/alternative


### Attachments

In [3]:
import mimetypes
from pathlib import Path

Path("report.csv").write_text("region,sales\nNorth,575\nSouth,443\n", encoding="utf-8")

def attach_file(message, path):
    path = Path(path)
    ctype, _ = mimetypes.guess_type(path.name)
    maintype, subtype = (ctype or "application/octet-stream").split("/", 1)
    message.add_attachment(path.read_bytes(), maintype=maintype, subtype=subtype, filename=path.name)

attach_file(msg, "report.csv")
attach_file(msg, "example.jpg")
print(msg.get_content_type())
for part in msg.iter_attachments():
    print(" attachment:", part.get_filename(), part.get_content_type())

multipart/mixed
 attachment: report.csv text/csv
 attachment: example.jpg image/jpeg


## 3. Sending

### Port 587 + STARTTLS (📗 flow, modernised)

In [ ]:
import os, smtplib, ssl

EMAIL = os.environ["EMAIL_USER"]
PASSWORD = os.environ["EMAIL_APP_PASSWORD"]
context = ssl.create_default_context()          # verifies the server's certificate

with smtplib.SMTP("smtp.gmail.com", 587, timeout=30) as server:   # `with` calls quit() for you
    server.ehlo()                 # greet the server (📗); optional, smtplib does it automatically when needed
    server.starttls(context=context)
    server.ehlo()
    server.login(EMAIL, PASSWORD)
    server.send_message(msg)      # ✅ send_message reads From/To/Cc from the headers
print("Sent!")

### Port 465 + SSL

In [ ]:
with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=ssl.create_default_context()) as server:
    server.login(EMAIL, PASSWORD)
    server.send_message(msg)

> ⚠️ **Old way (📗):**
> ```python
> msg = "Subject: " + subject + '\n' + message
> smtp_object.sendmail(from_address, to_address, msg)
> ```
> It works for simple ASCII text, but non-English characters, emojis, HTML and attachments break or show up garbled. Use `EmailMessage` + `send_message()`.

### Handling errors

```python
try:
    with smtplib.SMTP("smtp.gmail.com", 587, timeout=30) as s:
        s.starttls(context=ssl.create_default_context())
        s.login(EMAIL, PASSWORD)
        s.send_message(msg)
except smtplib.SMTPAuthenticationError:
    print("Login failed: check the app password")
except (smtplib.SMTPException, OSError) as e:
    print("Could not send:", e)
```

## 4. Testing without sending real mail

- Use a **fake SMTP server** that just prints messages: `pip install aiosmtpd`, then `python -m aiosmtpd -n -l localhost:8025` and connect with `smtplib.SMTP("localhost", 8025)`.
- ⚠️ The old built-in `smtpd` module (`python -m smtpd -n -c DebuggingServer`) was **removed in Python 3.12**.
- Services like Mailpit / MailHog give a local web inbox for testing.
- For production bulk/transactional mail, use a provider API (Amazon SES, SendGrid, Postmark, Resend...).

Save a message to disk to inspect it (it opens in any mail client):

In [4]:
Path("weekly_report.eml").write_bytes(bytes(msg))
print(Path("weekly_report.eml").stat().st_size, "bytes")

113958 bytes


## 5. Reading email with `imaplib` (📗)

In [ ]:
import imaplib, os

M = imaplib.IMAP4_SSL("imap.gmail.com")
M.login(os.environ["EMAIL_USER"], os.environ["EMAIL_APP_PASSWORD"])
print(M.list())                         # all mailboxes/folders
M.select("inbox")
typ, data = M.search(None, 'SUBJECT "weekly report"')
print(typ, data)                        # ('OK', [b'101 102 ...'])  message ids

### IMAP search keys (📗)

| Key | Meaning |
|---|---|
| `'ALL'` | all messages in the folder |
| `'BEFORE 01-Nov-2026'`, `'ON ...'`, `'SINCE ...'` | by date (format `DD-Mon-YYYY`) |
| `'FROM some_string'`, `'TO ...'`, `'CC ...'`, `'BCC ...'` | by address (or part of it) |
| `'SUBJECT string'`, `'BODY string'`, `'TEXT "with spaces"'` | by content |
| `'SEEN'` / `'UNSEEN'` | read / unread |
| `'ANSWERED'` / `'UNANSWERED'` | replied / not replied |
| `'DELETED'` / `'UNDELETED'` | deletion flag |

Combine keys: `M.search(None, '(FROM "boss@example.com" UNSEEN)')`, or `'OR FROM a@x.com FROM b@x.com'`.

In [ ]:
from email import message_from_bytes, policy

email_id = data[0].split()[-1]                         # the newest match
typ, msg_data = M.fetch(email_id, "(RFC822)")
received = message_from_bytes(msg_data[0][1], policy=policy.default)
print(received["From"], "|", received["Subject"])
body = received.get_body(preferencelist=("plain", "html"))
print(body.get_content()[:500])
M.close()
M.logout()

> ⚠️ **Old (📗):** `email.message_from_bytes(raw)` without `policy=policy.default` gives the legacy API (`get_payload(decode=True)`, manual `msg.walk()` loops). The modern policy gives `get_body()`, `get_content()` and `iter_attachments()`, and decodes headers like `=?utf-8?...` for you. The bootcamp also raises `imaplib._MAXLINE` (a private attribute) to avoid size limits; prefer fetching specific messages instead of everything.

### Parsing an email offline (the same API)

In [5]:
from email import message_from_bytes, policy

raw = Path("weekly_report.eml").read_bytes()
parsed = message_from_bytes(raw, policy=policy.default)
print("From:", parsed["From"])
print("To:", parsed["To"])
print("Subject:", parsed["Subject"])
print("Body:", parsed.get_body(preferencelist=("plain",)).get_content().strip())
for att in parsed.iter_attachments():
    data = att.get_content()
    print(f"Attachment {att.get_filename()} ({att.get_content_type()}, {len(data)} bytes)")

From: Python Bot <bot@example.com>
To: asha@example.com, ravi@example.com
Subject: Your weekly report 📊
Body: Hi team,

The weekly report is attached.

— Python Bot
Attachment report.csv (text/csv, 33 bytes)
Attachment example.jpg (image/jpeg, 83287 bytes)


## 6. Project ideas (📗 Exercise Ideas)
- Send yourself a daily email with the weather (Module 14 · 01 API) and your to-do list
- Email a CSV/Excel report generated by a script (Modules 11 & 14), scheduled with `cron`/Task Scheduler
- Alert yourself when a scraped price drops below a threshold (Module 14 · 02)
- Auto-download attachments from emails with a certain subject and save them to a folder
- A daily summary of unread emails from specific senders

## 📝 Summary
- ✅ Build messages with `EmailMessage` (`set_content`, `add_alternative`, `add_attachment`).
- Send with `smtplib.SMTP(host, 587)` + `starttls(context=ssl.create_default_context())`, or `SMTP_SSL(host, 465)`, inside `with`.
- Use app passwords and environment variables; never hard-code credentials.
- Read with `imaplib.IMAP4_SSL`, `search()`, `fetch()`; parse with `message_from_bytes(..., policy=policy.default)`.
- ⚠️ `smtpd` was removed in 3.12; use `aiosmtpd` or Mailpit for testing.

## 🏋️ Practice
1. Build an `EmailMessage` with an HTML table of 3 products and prices, plus a plain-text fallback. Print it.
2. Write `send_report(to, subject, body, attachments=[])` that builds and sends a message (test with a local fake server).
3. Parse `weekly_report.eml` and save every attachment to an `attachments/` folder.

<details><summary>Show solution for 3</summary>

```python
from email import message_from_bytes, policy
from pathlib import Path

msg = message_from_bytes(Path("weekly_report.eml").read_bytes(), policy=policy.default)
out = Path("attachments"); out.mkdir(exist_ok=True)
for att in msg.iter_attachments():
    (out / att.get_filename()).write_bytes(att.get_payload(decode=True))
print(list(out.iterdir()))
```
</details>